# Solutions — Capstone: TPC-H Supply Chain (Hard 17)

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

orders   = spark.table("samples.tpch.orders")
lineitem = spark.table("samples.tpch.lineitem")
customer = spark.table("samples.tpch.customer")
nation   = spark.table("samples.tpch.nation")
region   = spark.table("samples.tpch.region")
part     = spark.table("samples.tpch.part")
supplier = spark.table("samples.tpch.supplier")
partsupp = spark.table("samples.tpch.partsupp")


## Solution 1 — Customer Geography

In [ ]:
# Join customer → nation → region; compute per (region, market_segment)
result_1 = (
    customer
    .join(nation,  customer.c_nationkey == nation.n_nationkey)
    .join(region,  nation.n_regionkey  == region.r_regionkey)
    .groupBy(F.col("r_name").alias("region"), F.col("c_mktsegment").alias("market_segment"))
    .agg(
        F.count("c_custkey").alias("customer_count"),
        F.round(F.avg("c_acctbal"), 2).alias("avg_account_balance"),
    )
    .orderBy("region", "market_segment")
)


## Solution 2 — Revenue by Nation

In [ ]:
# Join orders → customer → nation → region; revenue = sum(o_totalprice)
result_2 = (
    orders
    .join(customer, orders.o_custkey    == customer.c_custkey)
    .join(nation,   customer.c_nationkey == nation.n_nationkey)
    .join(region,   nation.n_regionkey  == region.r_regionkey)
    .groupBy(F.col("n_name"), F.col("r_name").alias("region"))
    .agg(
        F.round(F.sum("o_totalprice"), 2).alias("total_revenue"),
        F.count("o_orderkey").alias("order_count"),
        F.round(F.avg("o_totalprice"), 2).alias("avg_order_value"),
    )
    .orderBy(F.col("total_revenue").desc())
)

## Solution 3 — Late Delivery Analysis

In [ ]:
# l_receiptdate > l_commitdate → late; group by (o_orderpriority, year)
result_3 = (
    lineitem
    .join(orders, lineitem.l_orderkey == orders.o_orderkey)
    .withColumn("year", F.year("l_shipdate"))
    .groupBy("o_orderpriority", "year")
    .agg(
        F.count("*").alias("total_items"),
        F.sum(F.when(F.col("l_receiptdate") > F.col("l_commitdate"), 1).otherwise(0)).alias("late_items"),
    )
    .withColumn("late_rate_pct",
        F.round(F.col("late_items") / F.col("total_items") * 100, 2)
    )
    .select("o_orderpriority", "year", "late_rate_pct")
    .orderBy("o_orderpriority", "year")
)


## Solution 4 — Supplier Ranking within Nation

In [ ]:
# revenue per supplier = sum(l_extendedprice * (1-l_discount)); rank within nation
w = Window.partitionBy("n_name").orderBy(F.col("total_revenue").desc())

result_4 = (
    lineitem
    .join(supplier, lineitem.l_suppkey == supplier.s_suppkey)
    .join(nation,   supplier.s_nationkey == nation.n_nationkey)
    .groupBy("s_name", "n_name")
    .agg(
        F.round(F.sum(F.col("l_extendedprice") * (1 - F.col("l_discount"))), 2).alias("total_revenue")
    )
    .withColumn("nation_rank", F.rank().over(w))
    .select("s_name", "n_name", "total_revenue", "nation_rank")
    .orderBy("n_name", "nation_rank")
)


## Solution 5 — Part Type Revenue by Market Segment

In [ ]:
# For each c_mktsegment, compute revenue per p_type; collect_set top_part_types
w = Window.partitionBy("c_mktsegment").orderBy(F.col("type_revenue").desc())

top_types = (
    lineitem
    .join(part,     lineitem.l_partkey == part.p_partkey)
    .join(orders,   lineitem.l_orderkey == orders.o_orderkey)
    .join(customer, orders.o_custkey == customer.c_custkey)
    .groupBy("c_mktsegment", "p_type")
    .agg(F.round(F.sum(F.col("l_extendedprice") * (1 - F.col("l_discount"))), 2).alias("type_revenue"))
    .withColumn("rnk", F.rank().over(w))
    .filter(F.col("rnk") <= 5)
)

result_5 = (
    top_types
    .groupBy("c_mktsegment")
    .agg(
        F.collect_set("p_type").alias("top_part_types"),
        F.round(F.sum("type_revenue"), 2).alias("segment_total_revenue"),
    )
    .orderBy("c_mktsegment")
)


## Solution 6 — Discount Impact by Shipping Mode

In [ ]:
# For each l_shipmode: revenue_with_discount, revenue_without_discount, discount_impact
result_6 = (
    lineitem
    .groupBy("l_shipmode")
    .agg(
        F.round(F.sum(F.col("l_extendedprice") * (1 - F.col("l_discount"))), 2).alias("revenue_with_discount"),
        F.round(F.sum("l_extendedprice"), 2).alias("revenue_without_discount"),
    )
    .withColumn("discount_impact",     F.round(F.col("revenue_without_discount") - F.col("revenue_with_discount"), 2))
    .withColumn("discount_impact_pct", F.round(F.col("discount_impact") / F.col("revenue_without_discount") * 100, 2))
    .select("l_shipmode", "revenue_with_discount", "revenue_without_discount", "discount_impact", "discount_impact_pct")
    .orderBy(F.col("discount_impact").desc())
)

## Solution 7 — Full Order Intelligence Report

In [ ]:
# High-value orders (o_totalprice > 200000); join orders → customer → nation;
# collect_set of supplier nations, part brands
supp_info = (
    lineitem
    .join(supplier, lineitem.l_suppkey == supplier.s_suppkey)
    .join(nation.alias("sn"), supplier.s_nationkey == F.col("sn.n_nationkey"))
    .join(part, lineitem.l_partkey == part.p_partkey)
    .groupBy("l_orderkey")
    .agg(
        F.count("*").alias("line_item_count"),
        F.round(F.sum(F.col("l_extendedprice") * (1 - F.col("l_discount"))), 2).alias("order_revenue"),
        F.collect_set(F.col("sn.n_name")).alias("supplier_nations"),
        F.collect_set("p_brand").alias("part_brands"),
    )
)

result_7 = (
    orders
    .filter(F.col("o_totalprice") > 200000)
    .join(customer, orders.o_custkey == customer.c_custkey)
    .join(nation,   customer.c_nationkey == nation.n_nationkey)
    .join(supp_info, orders.o_orderkey == supp_info.l_orderkey, how="left")
    .select(
        "o_orderkey",
        F.col("c_name").alias("customer_name"),
        F.col("n_name").alias("customer_nation"),
        "line_item_count",
        "order_revenue",
        "supplier_nations",
        "part_brands",
    )
    .orderBy(F.col("order_revenue").desc())
    .limit(50)
)